In [ ]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
# from datetime import date
import asyncio
import pandas as pd

In [ ]:
from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

In [ ]:
prices_to_use = "TRADES"

# ============================================================
# CHOICES:
#    "TRADES"
#    "MIDPOINT"
#    "BID"
#    "ASK"
#    "BID_ASK"
#    "ADJUSTED_LAST"
#    "HISTORICAL_VOLATILITY"
#    "OPTION_IMPLIED_VOLATILITY"
#    "FEE_RATE"
#    "REBATE_RATE"
#    "SCHEDULE"      
# ============================================================

In [ ]:
# ============================================================
# CHOOSE BETWEEN VARIABLE GROUPS     
# ============================================================

'''
lookback_period = "5 Y"
length_of_each_period = "1 day"
use_regular_trading_hours = True
'''

#'''
lookback_period = "i M"
length_of_each_period = "1 day"
use_regular_trading_hours = True
#'''

In [ ]:
# ============================================================
# INPUT SYMBOLS OR CONIDs AS A LIST
# ============================================================

symbols = ['AGG', 'BND', 'SCHZ', 'SPAB']
# conIds = [320106059, 641561653]

In [ ]:
async def get_historical_closes_df(contract_list, 
                                   lookback_period, 
                                   buffer_days=1,
                                   length_of_each_period='1 day',
                                   prices_to_use='TRADES',
                                   use_regular_trading_hours=True):

    df_list = []

    for contract in contract_list:

        sym = contract.symbol

        bars = await ibkr.ib.reqHistoricalDataAsync(
            contract=contract,
            endDateTime="",          # "" means now
            durationStr=lookback_period,
            barSizeSetting=length_of_each_period,
            whatToShow=prices_to_use,
            useRTH=use_regular_trading_hours,
            formatDate=1
        )

        df = pd.DataFrame([(bar.date, bar.close) for bar in bars], columns=["date", "close"])
        df['date'] = pd.to_datetime(df['date']).dt.date
        df[sym] = df['close']
        df = df.set_index("date")
        
        df_list.append(df[sym])
        
    big_df = pd.concat(df_list, axis=1)
    big_df = big_df.iloc[:-1]

    return big_df


In [ ]:
async def get_historical_prices_df(contract, 
                                   lookback_period, 
                                   length_of_each_period='1 day',
                                   prices_to_use='TRADES',
                                   use_regular_trading_hours=True):

    bars = await ibkr.ib.reqHistoricalDataAsync(
        contract=contract,
        endDateTime="",          # "" means now
        durationStr=lookback_period,
        barSizeSetting=length_of_each_period,
        whatToShow=prices_to_use,
        useRTH=use_regular_trading_hours,
        formatDate=1
    )

    
    df = pd.DataFrame(
        [
            {
                "date": bar.date,
                "open": bar.open,
                "high": bar.high,
                "low": bar.low,
                "close": bar.close,
                "volume": bar.volume,
            }
            for bar in bars
        ]
    )
    
    df['date'] = pd.to_datetime(df['date']).dt.date

    return df



In [ ]:

async def main():

    await start_ibkr()

    contract_list = []
    
    for sym in symbols:
    #for conId in conIds:

        contract = Stock(sym, 'SMART', 'USD')
        await ibkr.ib.qualifyContractsAsync(contract)

        contract_list.append(contract)

        # contract = await ibkr.contract_by_conId(conId)
        # contract = Future(symbol='BRR', lastTradeDateOrContractMonth='202606', exchange='CME', currency='USD')

    # df = await get_historical_closes_df(contract_list, lookback_period)
    df = await get_historical_prices_df(contract_list[0], lookback_period)

    print(df)

In [ ]:
# ============================================================
# MAIN
# ============================================================ 

await main()

#if __name__ == "__main__":
 #   asyncio.run(main())